# diagnostics.json 방출기 테스트 (작업지시서 §4.6)

앞의 두 산출물이 엑셀의 **내용**이었다면(칸 값 `cells.jsonl`, 칸끼리 연결 `references.json`),
이번 `diagnostics.json`은 파일의 **건강검진표**입니다.

파일을 훑어서 "이 통합문서는 이런 상태다"라는 **사실만** 한 장에 적습니다 —
어느 방법으로 열었나, 바깥 파일을 몇 개 참조하나, 이름 붙은 칸이 몇 개인가,
개인정보는 가렸나, 빈 칸을 참조하는 수식은 없나, 숨겨진 시트는 있나.

**중요한 규칙 하나**: "확인이 필요합니다" 같은 **의견·권고는 절대 안 씁니다**.
오직 관측한 숫자와 목록만. 판단은 나중에 어노테이터(LLM)의 몫입니다.

확인하는 것:
1. **V6 골든** — 감사계약 파일에서 지시서가 정한 정답 숫자가 나오는지
2. 이름 붙은 칸 이원 집계 + 깨진 참조·옛 경로 표시
3. 개인정보(이메일) 가리기 + 원문 유출 0건
4. 빈 칸을 참조하는 수식 골라내기
5. xls 파일 — 못 읽은 걸 "0개"가 아니라 "관찰 불가"로
6. 결정론(P2) — 두 번 만들어도 똑같은가 + 32종 전수

> 실행 전 커널이 이 프로젝트의 `.venv`인지 확인하세요.

In [1]:
# 0. 준비 — 경로 자동 판별과 import
from pathlib import Path
import hashlib, json, sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():   # tests/ 안에서 열었을 때
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from excel_to_skill.extractor import extract_workbook
from excel_to_skill.emit_diag import build_diagnostics, write_diagnostics

RAW = ROOT / "workingpaper_raw"   # 실조서 코퍼스 (git 밖)
OUT = ROOT / "tests" / "_output"  # 노트북 산출물 (git 밖)
OUT.mkdir(exist_ok=True)
print("프로젝트 루트:", ROOT)
print("코퍼스 파일 수:", len(list(RAW.glob("*.xls*"))))

프로젝트 루트: /home/shin/Project/Excel_to_Skill
코퍼스 파일 수: 32


## 1. V6 골든 — 정답 숫자와 대조

지시서 §4.6·V6은 감사계약 파일(`1100~1300`)의 진단표에서
**정확히 이 숫자들**이 나와야 한다고 못박아 뒀습니다. 하나라도 어긋나면 실패입니다.

- 외부 링크 **10개** (이 감사조서가 물어온 바깥 감사조서 파일 수)
- 이름 붙은 칸: 전역 **1363개** / 시트 전용 **594개** (둘을 갈라 세는 게 함정 7)
- 표본은 **20개까지만**, 전량 덤프는 나중에 `--full-names` 옵션으로

In [ ]:
gyeyak = next(RAW.glob("*1100~1300*"))
ir = extract_workbook(gyeyak)
doc = write_diagnostics(ir, OUT / "감사계약.diagnostics.json")

dn = doc["defined_names"]
print(f"원본: {gyeyak.name}")
print(f"로더 경로:        {doc['loader_path']}")
print(f"외부 링크 개수:   {doc['external_links']['count']}")
print(f"전역 이름:        {dn['global_total']}")
print(f"시트 전용 이름:   {dn['sheet_scoped_total']}")
print(f"표본 개수:        {len(dn['samples'])} (상한 {dn['sample_cap']})")
print(f"전량 덤프 포함?:  {dn['full_dump_present']}")

checks = {
    "외부 링크 == 10": doc["external_links"]["count"] == 10,
    "전역 == 1363": dn["global_total"] == 1363,
    "시트 전용 == 594": dn["sheet_scoped_total"] == 594,
    "표본 <= 20": len(dn["samples"]) <= 20,
    "sample_cap == 20": dn["sample_cap"] == 20,
    "full_dump_present == False": dn["full_dump_present"] is False,
}
print("\n── V6 골든 판정 ──")
for k, v in checks.items():
    print(f"  {'✓' if v else '✗'} {k}")
print("판정:", "통과 ✓" if all(checks.values()) else "실패 ✗")

원본: 감사조서서식_1100~1300 감사계약.xlsx
로더 경로:        openpyxl_normal
외부 링크 개수:   10
전역 이름:        1363
시트 전용 이름:   594
표본 개수:        20 (상한 20)
전량 덤프 포함?:  False

── V6 골든 판정 ──
  ✓ 외부 링크 == 10
  ✓ 전역 == 1363
  ✓ 시트 전용 == 594
  ✓ 표본 <= 20
  ✓ sample_cap == 20
  ✓ full_dump_present == False
판정: 통과 ✓


## 2. 이름 붙은 칸 — 깨진 참조·옛 경로 세기

엑셀은 칸이나 범위에 이름을 붙일 수 있습니다(예: `매출합계` = `Sheet1!$B$2`).
이 감사조서 서식엔 그런 이름이 1,957개(=1363+594)나 있는데, 오래된 서식이라
**탈이 난 이름**도 많습니다:

- **깨진 참조** : 값에 `#REF!`가 든 이름 — 원래 가리키던 칸이 지워진 것.
- **옛 파일 경로** : 값에 `C:\...` 같은 90년대 로컬 경로가 든 이름.

표본(20개)에는 이런 문제 이름을 **먼저** 보여줍니다(문제를 눈에 띄게). 각 표본은
이름·값 앞 60자·플래그를 담고, 값은 개인정보 규칙(P7)에 따라 미리 가립니다.

In [3]:
print(f"깨진 참조(#REF!) 든 이름:  {dn['broken_ref_count']}개")
print(f"옛 경로(C:\\ 등) 든 이름:    {dn['legacy_path_count']}개")

print("\n── 표본 20개 (문제 있는 것 먼저) ──")
for s in dn["samples"]:
    flag = ("[" + ",".join(s["flags"]) + "]") if s["flags"] else "[정상]"
    print(f"  {flag:<22} {s['name']:<18} = {s['value_head']}")

print("\n골든 대조:",
      "통과 ✓" if dn["broken_ref_count"] == 429 and dn["legacy_path_count"] == 32 else "실패 ✗",
      f"(깨진참조 429 / 옛경로 32 기대)")

깨진 참조(#REF!) 든 이름:  429개
옛 경로(C:\ 등) 든 이름:    32개

── 표본 20개 (문제 있는 것 먼저) ──
  [broken_ref]           __123Graph_D       = [1]IS!#REF!
  [broken_ref]           __123Graph_F       = '[2]2600'!#REF!
  [broken_ref]           __KEY2             = #REF!
  [broken_ref]           _Key3              = #REF!
  [broken_ref]           _MatInverse_In     = [1]IS!#REF!
  [broken_ref]           _MatInverse_Out    = '[2]2600'!#REF!
  [broken_ref]           _MatMult_B         = '[2]2600'!#REF!
  [legacy_path]          AccessDatabase     = "C:\생산판매\long98\9802장판원본.mdb"
  [broken_ref]           DATABASE1          = #REF!
  [legacy_path]          HTML_PathFile      = "C:\홈페이지\일위대가.htm"
  [legacy_path]          HTML1_12           = "C:\My Documents\98년\1월\영업현황\시험.htm"
  [legacy_path]          HTML10_12          = "C:\My Documents\98년\영업현황\일일현황-98.2.6.htm"
  [legacy_path]          HTML11_12          = "C:\My Documents\98년\영업현황\일일현황-98.2.12.htm"
  [legacy_path]          HTML12_12          = "C:\My Documents

## 3. 개인정보 가리기 (P7)

이 코퍼스는 90년대~2010년대 실제 감사조서라 **실명·이메일이 실제로 박혀 있습니다.**
진단표에 그대로 옮기면 안 되므로, 이메일은 로컬파트를 가립니다(`jaemins@` → `j***@`).

- `pii_suspects.emails_masked` : 이름·링크에서 찾아낸 이메일을 **가린 형태로** 목록화.
- 그리고 진단표 **어디에도 원문 이메일이 남지 않았는지**를 문자열 검색으로 최종 확인합니다.

> `legacy_paths_count`는 여기서 **0**이 맞습니다 — 이 값은 "본문 셀에서 나온 경로" 전용이고,
> 이름에서 나온 경로 32개는 위 2번의 `defined_names.legacy_path_count`에 이미 셌기 때문입니다
> (같은 값을 양쪽에 넣으면 이중 집계 — 코드 주석에 명시).

In [4]:
pii = doc["pii_suspects"]
print("가려진 이메일:", pii["emails_masked"])
print("본문 셀 경로 수(이 단계 0):", pii["legacy_paths_count"])

dump = json.dumps(doc, ensure_ascii=False)
leaked = "jaemins@" in dump   # 원본 로컬파트가 통째로 남았나?
print("\n진단표 전체에 원문 이메일 흔적:", "있음 ✗" if leaked else "없음 ✓")
print("판정:",
      "통과 ✓" if pii["emails_masked"] == ["j***@netian.co.kr"]
      and pii["legacy_paths_count"] == 0 and not leaked else "실패 ✗")

가려진 이메일: ['j***@netian.co.kr']
본문 셀 경로 수(이 단계 0): 0

진단표 전체에 원문 이메일 흔적: 없음 ✓
판정: 통과 ✓


## 4. 빈 칸을 참조하는 수식 골라내기

수식이 **아무것도 없는 빈 칸**을 가리키는 경우를 모읍니다. 예를 들어 `1200!B4`가
비어 있는 `1100!B4`(입력 슬롯)를 참조하면 — 지금은 계산 결과가 `00:00:00`처럼
엉뚱하게 나오지만, 사람이 값을 채우면 살아나는 자리입니다.

`references.json`의 화살표(edge)와 실제 칸 내용을 **맞대어** 찾습니다. 규칙 두 가지:
- **범위 참조는 제외** (칸 하나짜리만).
- 대상 주소가 원장에 **아예 없는 경우**(완전 빈 칸)도 포함.

In [5]:
blanks = doc["blank_source_formulas"]
print(f"빈 칸 참조 수식 {len(blanks)}건:\n")
for b in blanks:
    print(f"  {b['cell']:>8}  →  {b['source']:<8}  (가리킨 칸이 비어 있음)")
print("\n골든 대조:", "통과 ✓" if len(blanks) == 5 else "실패 ✗", "(5건 기대)")
print("※ 1300!D4→1200!D4 는 대상 1200!D4가 수식을 가져 '빈 칸' 아님 → 제외됨(정상)")

빈 칸 참조 수식 5건:

   1200!B4  →  1100!B4   (가리킨 칸이 비어 있음)
   1200!D4  →  1100!D4   (가리킨 칸이 비어 있음)
   1200!B5  →  1100!B5   (가리킨 칸이 비어 있음)
   1300!B4  →  1100!B4   (가리킨 칸이 비어 있음)
   1300!B5  →  1100!B5   (가리킨 칸이 비어 있음)

골든 대조: 통과 ✓ (5건 기대)
※ 1300!D4→1200!D4 는 대상 1200!D4가 수식을 가져 '빈 칸' 아님 → 제외됨(정상)


## 5. xls 파일 — 못 읽으면 "0"이 아니라 "관찰 불가"

옛 `.xls`는 라이브러리(xlrd)로 **외부 링크 목록을 못 읽습니다**. 이때 개수를 `0`으로
두면 "링크가 없다"는 거짓말이 됩니다. 그래서 `count`를 **`null`(관찰 불가)**로 둬서
"안 나온 것"과 "못 읽은 것"을 구분합니다(관찰 3상태 P6). `format_limitations`에 사유도 남깁니다.

In [6]:
xls_path = next(RAW.glob("*7540*"))   # .xls 파일
xdoc = build_diagnostics(extract_workbook(xls_path))
print(f"파일: {xls_path.name}")
print(f"로더 경로:        {xdoc['loader_path']}")
print(f"외부 링크 count:  {xdoc['external_links']['count']}   (None = 관찰 불가)")
print(f"숨김 시트:        {xdoc['hidden']['sheets']}")
print(f"format_limitations:\n  {xdoc['format_limitations']}")
print("\n판정:", "통과 ✓" if xdoc["external_links"]["count"] is None else "실패 ✗")

파일: 감사조서서식_7540 연결공시사항점검표 (KIFRS용)_2025.xls
로더 경로:        xlrd
외부 링크 count:  None   (None = 관찰 불가)
숨김 시트:        ['8400(2018)_수정 원본', '8400(2018)_업데이트(현 구조)']
format_limitations:
  xls(xlrd): 수식 원문 접근 불가 — 참조 그래프 관찰 불가(P6). 외부 링크 목록 관찰 불가.

판정: 통과 ✓


## 6. 결정론(P2) + 32종 전수

같은 파일을 두 번 진단해서 결과가 **글자 하나까지 똑같은지**(sha256 해시 비교),
그리고 코퍼스 32종 전체가 실패 없이 진단되는지 봅니다.
나중에 `verify` 명령의 V3(재현성)·V6(골든) 검사가 할 일의 예행연습입니다.

In [7]:
print(f"{'파일':<44} {'로더':<22} {'링크':>5} {'전역/시트':>11} {'빈칸':>4} 재현")
fails = []
for p in sorted(RAW.glob("*.xls*")):
    try:
        d1 = build_diagnostics(extract_workbook(p))
        d2 = build_diagnostics(extract_workbook(p))
        b1 = json.dumps(d1, ensure_ascii=False).encode()
        b2 = json.dumps(d2, ensure_ascii=False).encode()
        same = hashlib.sha256(b1).hexdigest() == hashlib.sha256(b2).hexdigest()
        if not same:
            fails.append((p.name, "결정론 불일치"))
        d = d1["defined_names"]
        gs = f"{d['global_total']}/{d['sheet_scoped_total']}"
        print(f"{p.name[:42]:<44} {d1['loader_path']:<22} "
              f"{str(d1['external_links']['count']):>5} {gs:>11} "
              f"{len(d1['blank_source_formulas']):>4} {'✓' if same else '✗'}")
    except Exception as e:
        fails.append((p.name, repr(e)))
        print(f"{p.name[:42]:<44} 실패: {e!r}")
print(f"\n실패 {len(fails)}건")
for name, err in fails:
    print(f"  ✗ {name}: {err}")

파일                                           로더                        링크       전역/시트   빈칸 재현


감사조서서식_01 조서파일 KIFRS.xlsx                    openpyxl_normal            0         0/0    0 ✓
감사조서서식_02 영구조서목록.xlsx                        openpyxl_normal            0         0/0    0 ✓


감사조서서식_03 일반조서목록 KIFRS.xlsx                  openpyxl_normal            0         0/0    0 ✓
감사조서서식_04 감사조서철 작성 및 보존.xlsx                 openpyxl_normal            0         0/0    0 ✓


감사조서서식_1100A_계약전 위험평가표 수임 2025.xlsx          openpyxl_normal            0       971/0  195 ✓


감사조서서식_1100~1300 감사계약.xlsx                   openpyxl_normal           10    1363/594    5 ✓


...

감사조서서식_7555 연결심리사항점검표.xlsx                   openpyxl_normal            0         0/0    0 ✓
감사조서서식_7555A_그룹감사 업무수행이사와 논의한 사항 및 자문한 사항    openpyxl_normal            0         0/0    0 ✓


감사조서서식_8100~8700 감사완결 2025.xlsx              openpyxl_normal            0    971/1228  688 ✓


감사조서서식_8400 공시사항점검표 (KIFRS용)_2025.xls        xlrd                    None         0/9    0 ✓


감사조서서식_8550 심리사항점검표.xlsx                     openpyxl_normal            0         0/0    0 ✓
감사조서서식_8550A_업무수행이사와 논의한 사항 및 자문한 사항 2025    openpyxl_normal            0         0/0    0 ✓


감사조서서식_9100~9800 내부회계관리제도감사 조서서식(예시).xlsx    openpyxl_read_only+xml_merge     2        12/0   59 ✓

실패 0건
